In [2]:
import pandas as pd
df = pd.read_csv('../lu2_datasets/Leescatalogus_Leesmateriaal.csv', sep=';')

**Kolommen verwijderen**

In [3]:
# Kolommen verwijderen die geen aanvullende informatie bevatten
df = df.drop(columns=["Unnamed: 3", "Genre_kopie"])

# Materiaal_id verwijderen, omdat Cataloguscode dezelfde unieke ID bevat
df = df.drop(columns=["Materiaal_id"])

**Tekstwaarden standaardiseren**

In [4]:
# Kleine letters en onnodige spaties verwijderen
df["Type_materiaal"] = df["Type_materiaal"].str.strip().str.lower()
df["Administratieve_prioriteit"] = df["Administratieve_prioriteit"].str.strip().str.lower()
df["Beschikbaar_formaat"] = df["Beschikbaar_formaat"].str.strip().str.lower()
df["Interessecluster"] = df["Interessecluster"].str.strip().str.lower()

df["Doelgroep"] = (
    df["Doelgroep"]
    .str.replace(" ", "", regex=False)
    .str.lower()
)

df["Doelgroep_extra"] = (
    df["Doelgroep_extra"]
    .str.replace(" ", "", regex=False)
    .str.lower()
)

# Taal standaardiseren
df["Taal"] = df["Taal"].replace({
    "NED": "Nederlands",
    "NL": "Nederlands",
    "nederlands": "Nederlands"
})

# Interne batchcode standaardiseren
df["Interne_batchcode"] = df["Interne_batchcode"].replace({
    "batch-2": "B2"
})

**Namen en auteurs corrigeren**

In [5]:
# Auteursnamen corrigeren
df["Auteur"] = df["Auteur"].replace({
    "Noami Alderman": "Naomi Alderman",
    "H. Mulisch": "Harry Mulisch",
    "Annejet vd Zijl": "Annejet van der Zijl"
})

# Meerdere auteurs op dezelfde manier noteren
df["Auteur"] = df["Auteur"].str.replace(
    "Hans en Monique Hagen",
    "Hans Hagen; Monique Hagen",
    regex=False
)

df["Auteur"] = (
    df["Auteur"]
    .str.replace(r"\s+en\s+", "; ", regex=True)
    .str.replace(r"\s*&\s*", "; ", regex=True)
    .str.replace(r"\s*/\s*", "; ", regex=True)
    .str.replace(r"\s*,\s*", "; ", regex=True)
)

**Thema's, onderwerpen en trefwoorden opschonen**

In [6]:
# Bronnen en URL's verwijderen uit trefwoorden
df["Trefwoorden"] = (
    df["Trefwoorden"]
    .str.replace(r"\s*[,/|;]\s*", ";", regex=True)
    .str.strip()
    .str.replace(r"\S*\.(?:nl|com)\S*", "", regex=True)
    .replace(["", "nan"], pd.NA)
)

# Bronnen verwijderen uit onderwerpen
df["Onderwerp"] = (
    df["Onderwerp"]
    .str.replace(r"\S*\.(?:nl|com)\S*", "", regex=True)
    .replace(["", "nan"], pd.NA)
)

# Thema's opschonen
df["Themas_boek"] = (
    df["Themas_boek"]
    .str.replace(r"\S*\.(?:nl|com)\S*", "", regex=True, case=False)
    .str.replace(r"\s*[,/|;]\s*", ";", regex=True)
    .str.strip(" ;")
    .replace(["", "nan"], pd.NA)
)

# Komma's vervangen door puntkomma's
kolommen = ["Themas_boek", "Onderwerp", "Trefwoorden"]

for kolom in kolommen:
    df[kolom] = df[kolom].str.replace(r",\s*", ";", regex=True)

**Numerieke waarden en datatypes opschonen**

In [7]:
# Aantal pagina's: negatieve en nulwaarden verwijderen
df["Aantal_paginas"] = (
    df["Aantal_paginas"]
    .mask(df["Aantal_paginas"] <= 0)
    .astype("Int64")
)

# Geschatte leestijd naar integer
df["Geschatte_leestijd_min"] = df["Geschatte_leestijd_min"].astype("Int64")

# Percentage omzetten naar float
df["Aandeel_lange_woorden_pct"] = (
    df["Aandeel_lange_woorden_pct"]
    .str.replace("%", "", regex=False)
    .astype(float)
)

# Onwaarschijnlijke gemiddelde woordlengtes als ontbrekend markeren
df.loc[df["Gem_woordlengte"] > 15, "Gem_woordlengte"] = pd.NA

**Categorieën samenvoegen en standaardiseren**

In [8]:
# Doelgroep en doelgroep_extra combineren
def combineer_doelgroep(rij):
    waarden = set()

    for kolom in ["Doelgroep", "Doelgroep_extra"]:
        waarde = rij[kolom]

        if pd.notna(waarde):
            waarden.update(waarde.lower().split("/"))

    volgorde = ["mbo", "nt2"]
    resultaat = [waarde for waarde in volgorde if waarde in waarden]

    return "/".join(resultaat) if resultaat else pd.NA

df["Doelgroep"] = df.apply(combineer_doelgroep, axis=1)
df = df.drop(columns=["Doelgroep_extra"])


# Type materiaal en genre aanpassen
nieuwe_genres = df["Type_materiaal"].str.split(" - ").str[1]

is_leeg = df["Genre"].isna()
staat_al_iets = df["Genre"].notna() & (df["Genre"] != "onbekend")

df.loc[
    df["Type_materiaal"].str.contains("boek -", na=False) & staat_al_iets,
    "Genre"
] = df["Genre"] + ";" + nieuwe_genres

df.loc[
    df["Type_materiaal"].str.contains("boek -", na=False) & is_leeg,
    "Genre"
] = nieuwe_genres

materiaal_mapping = {
    "online artikel": "artikel",
    "krant (papier)": "krant",
    "digitale krant": "krant",
    "boek - thriller": "boek",
    "boek - roman": "boek",
}

df["Type_materiaal"] = df["Type_materiaal"].replace(materiaal_mapping)

# Beschikbaar formaat standaardiseren
df["Beschikbaar_formaat"] = df["Beschikbaar_formaat"].replace("digitaal", "online")

formaat_is_onbekend = (
    (df["Beschikbaar_formaat"] == "onbekend")
    | df["Beschikbaar_formaat"].isna()
)

heeft_bron_url = (
    df["Bron_url"].notna()
    & (df["Bron_url"] != "onbekend")
    & (df["Bron_url"] != "")
)

df.loc[formaat_is_onbekend & heeft_bron_url, "Beschikbaar_formaat"] = "online"

**Ontbrekende en onjuiste gegevens corrigeren**

In [9]:
# Gegevensstatus standaardiseren
df["Gegevensstatus"] = df["Gegevensstatus"].replace({
    "Aangevuld (deels gesimuleerd)": "beschikbaar",
    "gecontroleerd": "niet beschikbaar",
    "In behandeling": "uitgeleend"
}).fillna("niet beschikbaar")


# Publicatiejaren corrigeren
df.loc[df["Titel"] == "De roadtrip", "Publicatiejaar"] = "2025"
df.loc[df["Titel"] == "Mijn naam is Edino", "Publicatiejaar"] = "2025"

df["Publicatiejaar"] = df["Publicatiejaar"].replace({
    "2099": pd.NA,
    "onbekend": pd.NA,
    "nan": pd.NA
})


# Ontbrekende doelgroep en niveaus aanvullen
boeken = [
    "Libello voor mannen",
    "Libello voor vrouwen",
    "De Startkrant",
    "De Startweekkrant"
]

df.loc[df["Titel"].isin(boeken), "Doelgroep"] = "nt2"
df.loc[df["Titel"].isin(boeken), "Niveau"] = "2F"

**Ontbrekende trefwoorden aanvullen vanuit omschrijvingen**

In [10]:
# Omschrijvingen zonder punt gebruiken om metadata aan te vullen
mask = ~df["Korte_omschrijving"].str.contains(r"\.", na=False)

kolommen = ["Themas_boek", "Onderwerp", "Trefwoorden"]

for kolom in kolommen:
    df.loc[mask, kolom] = (
        df.loc[mask, kolom].fillna("")
        + ";"
        + df.loc[mask, "Korte_omschrijving"]
    ).str.strip(";")

In [11]:
df["Geschatte_leestijd_min"].describe()

count         274.0
mean     377.343066
std      263.985609
min             5.0
25%           171.0
50%           366.0
75%           587.5
max           946.0
Name: Geschatte_leestijd_min, dtype: Float64

In [12]:
# aantal unieke en missende waarden en datatypes
print("Aantal rijen en kolommen:", df.shape)
overzicht = pd.DataFrame({
    "Aantal unieke waarden": df.nunique(),
    "Aantal missende waarden": df.isna().sum(),
    "Datatype": df.dtypes,
    "Aantal dubbele waarden": df.apply(
        lambda kolom: kolom.dropna().duplicated().sum()
    )
})

display(overzicht)

Aantal rijen en kolommen: (283, 29)


,Aantal unieke waarden,Aantal missende waarden,Datatype,Aantal dubbele waarden
Titel,277,0,str,6
Auteur,229,7,str,47
Korte_omschrijving,282,0,str,1
Type_materiaal,6,1,str,276
Niveau,4,2,str,277
Themas_boek,163,111,str,9
Genre,7,20,str,256
Onderwerp,117,111,str,55
Trefwoorden,163,111,str,9
Taal,1,0,str,282
